# DWM Lab 5 — Decision Tree Classification and Mutual Information

### Dataset: Online Retail

**Objective:**  
To implement a **Decision Tree Classification** model and calculate **Mutual Information** between the input features and the target variable.

### Dataset adaptation

The original lab uses a retail sales dataset containing `Advertising`, `Discount`, `Store Size`, `Sales` and a `Sales_Category` target.

The common **Online Retail** dataset does not contain those exact columns, so the same classification and feature-selection techniques are performed using:

- `Quantity`
- `UnitPrice`
- `Month`

The target `Sales_Category` is created from `SalesAmount`, where:

**SalesAmount = Quantity × UnitPrice**

The target has three classes:

- **Low**
- **Medium**
- **High**

`SalesAmount` itself is **not used as an input feature**, because it is used to create the target.

## 1. Import Required Libraries

Pandas is used for data manipulation, Matplotlib for visualization, and Scikit-learn for classification, evaluation and mutual information.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_selection import mutual_info_classif

## 2. Load the Dataset

Load the same `Online Retail.xlsx` dataset used in the previous DWM experiments.

In [ ]:
file_path = "Online Retail.xlsx"
df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Data Preprocessing

The dataset is cleaned before classification.

Steps:
1. Convert `InvoiceDate` to datetime.
2. Remove cancelled invoices.
3. Remove transactions with non-positive quantity or unit price.
4. Calculate `SalesAmount`.
5. Extract the month from `InvoiceDate`.
6. Create the three-class `Sales_Category` target using sales-value tertiles.
7. Remove missing values from the selected features.

In [ ]:
# Convert date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')].copy()

# Keep valid transactions
df = df[
    (df['Quantity'] > 0) &
    (df['UnitPrice'] > 0)
].copy()

# Calculate sales amount
df['SalesAmount'] = df['Quantity'] * df['UnitPrice']

# Extract month
df['Month'] = df['InvoiceDate'].dt.month

# Create Low / Medium / High categories using tertiles
q1 = df['SalesAmount'].quantile(1/3)
q2 = df['SalesAmount'].quantile(2/3)

def create_category(value):
    if value <= q1:
        return 'Low'
    elif value <= q2:
        return 'Medium'
    else:
        return 'High'

df['Sales_Category'] = df['SalesAmount'].apply(create_category)

# Select model features
model_df = df[['Quantity', 'UnitPrice', 'Month', 'Sales_Category']].dropna()

print("Cleaned dataset shape:", model_df.shape)
print("\nCategory thresholds:")
print("Low    :", "<=", q1)
print("Medium :", ">", q1, "and <=", q2)
print("High   :", ">", q2)

display(model_df.head())

## 4. Examine Class Distribution

Check how many transactions belong to each sales category.

In [ ]:
category_counts = model_df['Sales_Category'].value_counts()

print(category_counts)

plt.figure(figsize=(7, 5))
category_counts.plot(kind='bar')
plt.title('Sales Category Distribution')
plt.xlabel('Sales Category')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Prepare Training and Testing Data

The input variables are:

- `Quantity`
- `UnitPrice`
- `Month`

The target variable is:

- `Sales_Category`

The data is split into **80% training** and **20% testing** data.

In [ ]:
X = model_df[['Quantity', 'UnitPrice', 'Month']]
y = model_df['Sales_Category']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 6. Train the Decision Tree Classifier

A Decision Tree classifier is trained to predict whether a transaction belongs to the Low, Medium or High sales category.

`criterion='entropy'` is used to build the tree using information gain.

In [ ]:
dt_model = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

print("Decision Tree trained successfully.")

## 7. Evaluate the Decision Tree

The classifier is evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=['Low', 'Medium', 'High'])
print(cm)

## 8. Visualize the Confusion Matrix

The confusion matrix shows the number of correctly and incorrectly classified transactions for each sales category.

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Low', 'Medium', 'High']
)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax)
ax.set_title('Decision Tree Confusion Matrix')
plt.tight_layout()
plt.show()

## 9. Visualize the Decision Tree

The trained Decision Tree is displayed below.

Each node shows the splitting condition, impurity, number of samples and predicted class.

In [ ]:
plt.figure(figsize=(20, 10))

plot_tree(
    dt_model,
    feature_names=X.columns,
    class_names=['High', 'Low', 'Medium'],
    filled=True,
    rounded=True,
    fontsize=9
)

plt.title('Decision Tree Classifier')
plt.show()

## 10. Feature Importance

Decision Trees provide the importance of each feature used for classification.

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

display(feature_importance)

plt.figure(figsize=(8, 5))
plt.bar(
    feature_importance['Feature'],
    feature_importance['Importance']
)
plt.title('Decision Tree Feature Importance')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## 11. Mutual Information

Mutual Information measures the dependency between an input feature and the target variable.

A higher Mutual Information value indicates a stronger statistical dependency between the feature and the target.

Formula conceptually:

**MI(X;Y) = Σ P(x,y) log[P(x,y) / (P(x)P(y))]**

In [ ]:
# Calculate mutual information
mi_values = mutual_info_classif(
    X,
    y,
    random_state=42
)

mi_df = pd.DataFrame({
    'Feature': X.columns,
    'Mutual Information': mi_values
}).sort_values('Mutual Information', ascending=False)

display(mi_df)

## 12. Visualize Mutual Information

Compare the Mutual Information values of the input features.

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(
    mi_df['Feature'],
    mi_df['Mutual Information']
)
plt.title('Mutual Information of Features')
plt.xlabel('Feature')
plt.ylabel('Mutual Information')
plt.tight_layout()
plt.show()

## 13. Predict a New Transaction

Use the trained Decision Tree to classify a new transaction.

Example:
- Quantity = 10
- Unit Price = 5
- Month = 12

In [ ]:
new_transaction = pd.DataFrame({
    'Quantity': [10],
    'UnitPrice': [5],
    'Month': [12]
})

prediction = dt_model.predict(new_transaction)

print("New Transaction:")
display(new_transaction)

print("Predicted Sales Category:", prediction[0])

## Conclusion

A Decision Tree Classification model was implemented on the Online Retail dataset.

The experiment included:

- Data preprocessing
- Creation of Low, Medium and High sales categories
- 80:20 train-test split
- Decision Tree classification using entropy
- Accuracy, precision, recall and F1-score
- Confusion matrix
- Decision Tree visualization
- Feature importance
- Mutual Information analysis
- Prediction for a new transaction

The same Online Retail dataset can be reused for the remaining DWM experiments.